# Run ARC Phase 1 on MedQA-US (USMLE)

Clones the ARC repository from GitLab, installs dependencies, creates `.env` from `.env.example`, and runs the Phase 1 conflict pipeline on **MedQA-USMLE 4-option** (`GBaker/MedQA-USMLE-4-options`).

## 1. Runtime Parameters

Edit these values before running the notebook if needed.

In [ ]:
from pathlib import Path

REPO_URL = "https://gitlab.com/beryl.hoe/arc.git"
PROJECT_DIR = Path("/content/arc")
OUTPUT_FILENAME = "medqa_us_scenarios.jsonl"
N_QUESTIONS = None  # None -> use DEFAULT_N_QUESTIONS from .env
PREPARE_STATPEARLS = True
FORCE_RECLONE = False

## 2. Install System Packages

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-21-jdk-headless git git-lfs wget > /dev/null
!git lfs install

## 3. Clone the Repository

In [ ]:
import shutil
import subprocess

if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists():
    print(f"Repository already exists at {PROJECT_DIR}; pulling latest changes.")
    subprocess.run(["git", "pull"], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)

print(f"Project directory: {PROJECT_DIR}")

## 4. Install Python Dependencies

In [ ]:
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements.txt")],
    check=True,
)

## 5. Create and Load `.env`

The repository only tracks `.env.example`. This cell copies it to `.env` when needed.

In [ ]:
import os
import shutil

env_path = PROJECT_DIR / ".env"
env_example_path = PROJECT_DIR / ".env.example"

if not env_path.exists():
    if not env_example_path.exists():
        raise FileNotFoundError(f"Missing both {env_path} and {env_example_path}")
    shutil.copyfile(env_example_path, env_path)
    print(f"Created {env_path} from {env_example_path}")
else:
    print(f"Using existing {env_path}")

def load_dotenv(path):
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")

load_dotenv(env_path)
for key in ["USE_DRIVE", "DRIVE_ROOT", "LOCAL_ROOT", "LOCAL_EMBED_DIR", "HF_HOME",
            "HF_TOKEN", "CORPUS_NAME", "TOP_K", "RETRIEVERS", "DEFAULT_N_QUESTIONS",
            "NLI_MODEL", "PKE_MODEL", "N_PROBES"]:
    val = os.environ.get(key, "")
    print(f"{key}={'<set>' if key == 'HF_TOKEN' and val else val}")

## 6. Mount Google Drive

Drive must be mounted from the notebook kernel, not from the CLI subprocess.

In [ ]:
use_drive = os.environ.get("USE_DRIVE", "false").lower() in {"1", "true", "yes", "y", "on"}
if use_drive:
    from google.colab import drive
    drive.mount("/content/drive")
else:
    print("USE_DRIVE is false; skipping Google Drive mount.")

## 7. Optional Hugging Face Authentication

In [ ]:
hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    subprocess.run(["huggingface-cli", "login", "--token", hf_token], check=True)
else:
    print("HF_TOKEN is empty; skipping Hugging Face login.")

## 8. Prepare StatPearls

StatPearls chunks must be built from the MedRAG working directory before BM25 is initialised.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))

from src.config import load_config
from src.environment import is_statpearls_ready, prepare_runtime, prepare_statpearls

CONFIG = load_config()
prepare_runtime(CONFIG, mount_drive=False)

if PREPARE_STATPEARLS:
    prepare_statpearls(CONFIG)

print(f"statpearls_ready={is_statpearls_ready(CONFIG)}")

## 9. Preflight Check

In [ ]:
preflight = subprocess.run(
    [sys.executable, "-c",
     "from src.config import CONFIG; import src.cli; print('python ok'); print(CONFIG.root_dir); print(CONFIG.output_dir)"],
    cwd=PROJECT_DIR, text=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE,
)
print("--- STDOUT ---"); print(preflight.stdout)
print("--- STDERR ---"); print(preflight.stderr)
if preflight.returncode != 0:
    raise RuntimeError(f"Preflight failed with exit code {preflight.returncode}")

## 10. Run the Pipeline on MedQA-US

In [ ]:
default_n_questions = int(os.environ.get("DEFAULT_N_QUESTIONS", "10"))
n_questions = N_QUESTIONS if N_QUESTIONS is not None else default_n_questions

cmd = [sys.executable, "-u", "-m", "src.cli",
       "--dataset", "medqa-us",
       "--n-questions", str(n_questions),
       "--output", OUTPUT_FILENAME]

if PREPARE_STATPEARLS:
    cmd.append("--prepare-statpearls")

print("Running:", " ".join(cmd))
process = subprocess.Popen(
    cmd, cwd=PROJECT_DIR, text=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"Pipeline failed with exit code {returncode}")

## 11. Inspect Output

In [ ]:
use_drive = os.environ.get("USE_DRIVE", "false").lower() in {"1", "true", "yes", "y", "on"}
root_dir = Path(os.environ["DRIVE_ROOT"] if use_drive else os.environ["LOCAL_ROOT"])
output_path = root_dir / "outputs" / OUTPUT_FILENAME

print(f"Output path: {output_path}")
if output_path.exists():
    print(f"Size: {output_path.stat().st_size / 1024:.1f} KiB")
    with output_path.open("r", encoding="utf-8") as handle:
        first_line = handle.readline().strip()
    print(first_line[:2000])
else:
    print("Output file was not found.")